# A4: Text Preprocessing and OOV Handling

Build a preprocessing pipeline: tokenization, sentence segmentation, stop-word removal, stemming and lemmatization, text normalization and true casing. Identify and handle out-of-vocabulary (OOV) words with unknown-token replacement. Compare the text before and after preprocessing.

**Dataset:** IMDB Movie Reviews (5,000 sampled from 50,000) and the Twitter samples in NLTK (10,000 tweets, noisy text)

In [2]:
# nltk   : tokenizers, stop words, stemmer, lemmatizer, Twitter corpus
# pandas : tables
!pip install -q nltk pandas

In [1]:
import re
from collections import Counter
import pandas as pd
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "twitter_samples"]:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize, TweetTokenizer
from nltk.corpus import stopwords, twitter_samples
from nltk.stem import PorterStemmer, WordNetLemmatizer

## Load the data

In [4]:
imdb = pd.read_csv("https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv")
reviews = imdb["review"].sample(5000, random_state=1).tolist()
tweets = twitter_samples.strings("positive_tweets.json") + twitter_samples.strings("negative_tweets.json")
import random
random.Random(1).shuffle(tweets)                     # mix positive and negative tweets
print(len(reviews), "reviews,", len(tweets), "tweets")

5000 reviews, 10000 tweets


In [6]:
review = reviews[3]
tweet = tweets[1024]                                 # a noisy one: mentions, hashtags, a link, an emoticon
print(review[:400]); print(); print(tweet)

Ridiculous horror film about a wealthy man (John Carradine) dying and leaving everything to his four children, and his servants to be divided up equally. One condition--they must spend one week in his estate to get the money. And if any of them die, the others get more. Guess what happens next.... <br /><br />I saw a brand new print of this film on cable. The colors were bright and vivid and the h

I'm thinking garden party :)... #N04JS Buy 6 bottles of wine save 25%!! @chrisyamahar201 @sainsburys #prosecco http://t.co/M3kvcw24N7


## 1. Sentence segmentation

In [7]:
sents = sent_tokenize(review.replace("<br />", " "))
print(len(sents), "sentences")
for s in sents[:4]:
    print("-", s)

15 sentences
- Ridiculous horror film about a wealthy man (John Carradine) dying and leaving everything to his four children, and his servants to be divided up equally.
- One condition--they must spend one week in his estate to get the money.
- And if any of them die, the others get more.
- Guess what happens next....


## 2. Tokenization: normal text vs tweets

In [8]:
print(word_tokenize(tweet))
print(TweetTokenizer().tokenize(tweet))      # keeps @mentions, #hashtags, :) and URLs together

['I', "'m", 'thinking', 'garden', 'party', ':', ')', '...', '#', 'N04JS', 'Buy', '6', 'bottles', 'of', 'wine', 'save', '25', '%', '!', '!', '@', 'chrisyamahar201', '@', 'sainsburys', '#', 'prosecco', 'http', ':', '//t.co/M3kvcw24N7']
["I'm", 'thinking', 'garden', 'party', ':)', '...', '#N04JS', 'Buy', '6', 'bottles', 'of', 'wine', 'save', '25', '%', '!', '!', '@chrisyamahar201', '@sainsburys', '#prosecco', 'http://t.co/M3kvcw24N7']


## 3. Text normalization

In [9]:
def normalize(text):
    text = text.replace("<br />", " ")                    # HTML line breaks in IMDB
    text = re.sub(r"https?://\S+", " ", text)             # links
    text = re.sub(r"@\w+", " ", text)                     # @mentions
    text = re.sub(r"#", "", text)                         # keep the hashtag word, drop the #
    text = re.sub(r"(\w)\1{2,}", r"\1\1", text)          # soooo -> soo
    text = re.sub(r"\d+", " ", text)                      # numbers
    text = text.lower()
    text = text.replace("can't", "can not").replace("won't", "will not")
    text = re.sub(r"n't\b", " not", text)                  # didn't -> did not, keeps the negation
    return text

print(normalize(tweet))
print(normalize("I loooooved it!!! 10/10 #BestMovie @friend https://t.co/xyz"))

i'm thinking garden party :)... n js buy   bottles of wine save  %!!     prosecco  
i looved it!!!  /  bestmovie    


## 4. Stop-word removal (keeping negations)

In [10]:
swords = set(stopwords.words("english")) - {"not", "no", "nor", "never"}   # negations change sentiment, keep them
tokens = [t for t in TweetTokenizer().tokenize(normalize(tweet)) if t.isalpha()]
print("before:", tokens)
print("after: ", [t for t in tokens if t not in swords])

before: ['thinking', 'garden', 'party', 'n', 'js', 'buy', 'bottles', 'of', 'wine', 'save', 'prosecco']
after:  ['thinking', 'garden', 'party', 'n', 'js', 'buy', 'bottles', 'wine', 'save', 'prosecco']


## 5. Stemming and lemmatization

In [11]:
ps, wnl = PorterStemmer(), WordNetLemmatizer()
words = ["movies", "running", "studies", "better", "watched", "happily"]
pd.DataFrame({"word": words, "stem": [ps.stem(w) for w in words],
              "lemma (noun)": [wnl.lemmatize(w) for w in words], "lemma (verb)": [wnl.lemmatize(w, "v") for w in words]})

,word,stem,lemma (noun),lemma (verb)
0,movies,movi,movie,movies
1,running,run,running,run
2,studies,studi,study,study
3,better,better,better,better
4,watched,watch,watched,watch
5,happily,happili,happily,happily


## 6. True casing

Lowercasing loses information ("us" vs "US", "apple" vs "Apple"). A simple truecaser learns, from text where case is reliable, the most common way each word is written in the middle of a sentence.

In [12]:
case_counts = {}
for r in reviews:
    for s in sent_tokenize(r.replace("<br />", " ")):
        for w in word_tokenize(s)[1:]:                     # skip the first word: it is always capitalised
            if w.isalpha():
                case_counts.setdefault(w.lower(), Counter())[w] += 1

def truecase(text):
    out = []
    for i, w in enumerate(word_tokenize(text.lower())):
        best = case_counts.get(w, Counter({w: 1})).most_common(1)[0][0]
        out.append(best.capitalize() if i == 0 else best)
    return " ".join(out)

print(truecase("i watched the new marvel movie in india with tom and the us team"))

I watched the new marvel movie in India with Tom and the us team


## 7. The full pipeline, applied to both datasets

In [13]:
def preprocess(text):
    tokens = TweetTokenizer().tokenize(normalize(text))
    tokens = [t for t in tokens if t.isalpha() and t not in swords]
    return [wnl.lemmatize(t) for t in tokens]

print(preprocess(tweet))
print(preprocess(review)[:25])

['thinking', 'garden', 'party', 'n', 'j', 'buy', 'bottle', 'wine', 'save', 'prosecco']
['ridiculous', 'horror', 'film', 'wealthy', 'man', 'john', 'carradine', 'dying', 'leaving', 'everything', 'four', 'child', 'servant', 'divided', 'equally', 'one', 'must', 'spend', 'one', 'week', 'estate', 'get', 'money', 'die', 'others']


In [14]:
raw_tok = lambda text: text.split()                    # "before": just split on spaces
data = {"IMDB": reviews, "Twitter": tweets}
before = {k: [raw_tok(t) for t in v] for k, v in data.items()}
after = {k: [preprocess(t) for t in v] for k, v in data.items()}
print("done")

done


## 8. Out-of-vocabulary words

Build the vocabulary from the first 80% of texts (words seen at least twice), then check the last 20%: any word not in the vocabulary is OOV and becomes `<UNK>`.

In [15]:
def oov_stats(docs):
    cut = int(0.8 * len(docs))
    train, test = docs[:cut], docs[cut:]
    counts = Counter(w for d in train for w in d)
    vocab = {w for w, c in counts.items() if c >= 2}
    test_words = [w for d in test for w in d]
    oov = [w for w in test_words if w not in vocab]
    return vocab, len(oov) / len(test_words), Counter(oov)

def replace_unk(doc, vocab):
    return [w if w in vocab else "<UNK>" for w in doc]

vocab, rate, oov_words = oov_stats(after["Twitter"])
print(f"Twitter after preprocessing: OOV rate {rate:.1%}")
print("most common OOV words:", oov_words.most_common(10))
print(replace_unk(after["Twitter"][-5], vocab))

Twitter after preprocessing: OOV rate 19.2%
most common OOV words: [('retweeted', 5), ('vine', 4), ('hill', 4), ('germany', 4), ('likeforfollow', 4), ('sleepy', 4), ('artist', 4), ('letting', 4), ('master', 3), ('fault', 3)]
['yes', '<UNK>', 'definitely', 'something']


## 9. Before vs after

In [16]:
rows = []
for name in data:
    for stage, docs in [("before", before[name]), ("after", after[name])]:
        vocab, rate, _ = oov_stats(docs)
        rows.append({"dataset": name, "stage": stage,
                     "tokens per text": round(sum(map(len, docs)) / len(docs), 1),
                     "vocabulary size": len(set(w for d in docs for w in d)),
                     "OOV rate in test part": f"{rate:.1%}"})
pd.DataFrame(rows)

,dataset,stage,tokens per text,vocabulary size,OOV rate in test part
0,IMDB,before,230.3,100529,8.8%
1,IMDB,after,116.4,32905,5.2%
2,Twitter,before,11.6,28492,24.9%
3,Twitter,after,5.4,10420,19.2%


In [16]:
i = 0
print("BEFORE:", " ".join(before["Twitter"][i]))
print("AFTER: ", " ".join(after["Twitter"][i]))

BEFORE: why can't i go to sleep at a normal time :(
AFTER:  not go sleep normal time


## Conclusion
- Preprocessing shrinks the vocabulary a lot: case, punctuation, links, mentions, numbers and word forms no longer create separate words.
- Fewer distinct words means fewer unknown words in new text, so the OOV rate drops on both datasets; tweets stay noisier than reviews.
- Remaining OOV words (names, slang, typos) are replaced by `<UNK>` so a model can still process them.
- Choices depend on the task: negations were kept because they matter for sentiment; true casing matters for tasks like NER.